In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.influence_func_cp import (
    InfluenceFunctionConformalPredictor,
)

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [7]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs",
    loss_name=loss_name,
    loss_params=loss_params,
)

Instantiate region predictor

In [8]:
conformal_predictor = InfluenceFunctionConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [9]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [10]:
prediction_regions

[{'upper': [-1.7238070906343232,1.730283988313511],
  'lower': [-1.7236906375573904,1.7301666703906395]},
 {'upper': [-1.7245785513705936,1.7295123722739927],
  'lower': [-1.7244619806304582,1.7293951722284358]},
 {'upper': [-1.7248479755563053,1.729229975866579],
  'lower': [-1.7247313442433134,1.7291128369438074]},
 {'upper': [-1.726961338541078,1.7271143137860656],
  'lower': [-1.7268443962304372,1.7269974861004316]},
 {'upper': [-1.7232841680917559,1.7308073398444663],
  'lower': [-1.7231677911492849,1.7306899456208742]},
 {'upper': [-1.7244327909941173,1.7296396570306831],
  'lower': [-1.724316231332678,1.7295224465648555]},
 {'upper': [-1.7229791623575048,1.7311059641278326],
  'lower': [-1.7228628355539293,1.730988519904139]},
 {'upper': [-1.7246678661164674,1.7294123964354258],
  'lower': [-1.7245512761101227,1.729295216079867]},
 {'upper': [-1.7260281082067777,1.7280408726293257],
  'lower': [-1.725911298588844,1.7279239124806491]},
 {'upper': [-1.7245890243578166,1.7295003780

In [11]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  0.928


In [12]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.928
